In [7]:
import numpy as np
datapath = "../../Preprocessing/Encoded_Data/mopb_embeddings.npz"
data = np.load(datapath)
embeddings = data["embeddings"]
labels = data["labels"]

In [8]:
embeddings.shape

(1300, 2, 320)

In [9]:
mid_layer  = embeddings[:, 0, :]
last_layer = embeddings[:, 1, :]
embeddings_pooled = embeddings.mean(axis=1)
print("Mid layer:", mid_layer[0][:10])
print("Last layer:", last_layer[0][:10])
print("Pooled: ", embeddings_pooled[0][:10])

Mid layer: [ 0.28065887 -0.27746308 -0.21100964  0.5146927   0.24426301  0.40748656
  0.2249523  -0.14797825 -0.07932167 -0.31928954]
Last layer: [ 0.06201977 -0.41397864  0.0554308   0.16810417  0.02938422  0.03182533
 -0.24222983  0.59170896  1.3484923  -0.51699805]
Pooled:  [ 0.17133932 -0.34572086 -0.07778942  0.34139845  0.13682361  0.21965595
 -0.00863877  0.22186536  0.6345853  -0.4181438 ]


In [10]:
from sklearn.cluster import HDBSCAN
from sklearn.metrics import v_measure_score

best_score = 0
best_params = {}
best_results = {}
embedding_options = {"Mid layer": mid_layer, "Last layer": last_layer, "Pooled": embeddings_pooled}

for min_cluser_size in [5, 10, 20, 30, 50]:
    for min_samples in [1, 5, 10]:
        for metric in ["euclidean", "manhattan"]:
            for embedding in embedding_options:
                clusterer = HDBSCAN(
                min_cluster_size=min_cluser_size,
                min_samples=min_samples,
                metric=metric,
                copy=True
                )
                hdb_labels = clusterer.fit_predict(embedding_options[embedding])

                if len(set(hdb_labels)) == 1: # only noise
                    continue

                score = v_measure_score(labels, hdb_labels)

                if score > best_score:
                    best_score = score
                    best_params = {
                        'min_cluster_size': min_cluser_size,
                        'min_samples': min_samples,
                        'metric': metric,
                        'embedding': embedding
                    }
                    best_results = {
                        'Number of cluster': len(set(hdb_labels)),
                        'Number of noise points': sum(hdb_labels == -1)
                    }

print("Best score:", best_score)
print("Best parameters:", best_params)
print("Best result:", best_results)

Best score: 0.6098255516201622
Best parameters: {'min_cluster_size': 20, 'min_samples': 1, 'metric': 'manhattan', 'embedding': 'Last layer'}
Best result: {'Number of cluster': 8, 'Number of noise points': np.int64(158)}
